## Лабораторная работа 4
### Хеширование

№ 1
(5 балла)
Возьмите реализацию класса HashTable из лекционных материалов и выполните
следующие доработки:
1. Реализуйте квадратичное пробирование как технику повторного хеширования.
2. Реализуйте работу с функцией len (переопределите метод __len__).
3. Реализуйте работу оператора in (переопределите метод __contains__).
4. Переделайте метод put таким образом, чтобы таблица автоматически меняла размер,
когда загрузочный фактор становится больше значения 0.7. Размер должен
увеличиваться примерно в два раза до ближайшего подходящего простого числа.
5. Реализуйте работу оператора del (переопределите метод __delitem__) для удаления
элемента таблицы. Таблица должна автоматически менять размер, когда
загрузочный фактор становится меньше значения 0.2. Размер должен уменьшаться
примерно в два раза до ближайшего подходящего простого числа.
Все выполненные доработки должны быть протестированы.

In [1]:
import math

In [63]:
def next_prime(n):
    def is_prime(num):
        if num < 2:
            return False
        for i in range(2, int(num ** 0.5) + 1):
            if num % i == 0:
                return False
        return True

    candidate = n + 1
    while True:
        if is_prime(candidate):
            return candidate
        candidate += 1


class HashTable:
    def __init__(self, size=11):
        self.size = size               # текущий размер таблицы (число слотов)
        self.slots = [None] * size     # список ключей
        self.data = [None] * size      # список значений
        self.count = 0                 # количество элементов

    def hashfunction(self, key, size):
        return key % size

    def rehash(self, hashvalue, size, attempt):
        return (hashvalue + attempt ** 2) % size

    def put(self, key, data):

        if self.count / self.size > 0.7:
            self.resize()

        hashvalue = self.hashfunction(key, self.size)
        attempt = 0

        while True:
            new_index = self.rehash(hashvalue, self.size, attempt)

            # Пустой слот — вставляем новый элемент
            if self.slots[new_index] is None:
                self.slots[new_index] = key
                self.data[new_index] = data
                self.count += 1
                return

            # Уже существующий ключ — заменяем значение
            if self.slots[new_index] == key:
                self.data[new_index] = data
                return

            # Иначе пробуем следующий слот
            attempt += 1
            if attempt > self.size:
                raise RuntimeError("Хеш-таблица переполнена")

    def get(self, key):
        hashvalue = self.hashfunction(key, self.size)
        attempt = 0

        while True:
            new_index = (hashvalue + attempt ** 2) % self.size

            # Пустой слот — значит, ключа нет
            if self.slots[new_index] is None:
                return None

            # Найден нужный ключ
            if self.slots[new_index] == key:
                return self.data[new_index]

            attempt += 1
            if attempt > self.size:
                return None

    def resize(self):
        old_slots = self.slots
        old_data = self.data

        if self.count / self.size > 0.7:
            new_size = next_prime(self.size * 2)
        elif self.count / self.size < 0.2:
            new_size = next_prime(self.size // 2)

        self.size = new_size
        self.slots = [None] * new_size
        self.data = [None] * new_size
        self.count = 0

        for i in range(len(old_slots)):
            if old_slots[i] is not None:
                self.put(old_slots[i], old_data[i])

    def __getitem__(self, key):
        return self.get(key)

    def __setitem__(self, key, data):
        self.put(key, data)

    def __len__(self):
        return self.count

    def __contains__(self, key):
        if self.get(key) is not None:
            return True

    def __delitem__(self, key):
        if self.count / self.size < 0.2:
            self.resize()
        hashvalue = self.hashfunction(key, self.size)
        attempt = 0

        while True:
            index = (hashvalue + attempt ** 2) % self.size

            if self.slots[index] is None:
                print(f"Ключ {key} не найден — нечего удалять.")
                return

            if self.slots[index] == key:
                self.slots[index] = None
                self.data[index] = None
                self.count -= 1
                break

            attempt += 1
            if attempt > self.size:
                print(f"Ключ {key} не найден — нечего удалять.")
                return


def test_hash_table():
    H = HashTable()

    # 1. Вставка элементов
    print("Добавляем элементы:")
    H[54] = "cat"
    H[26] = "dog"
    H[93] = "lion"
    H[17] = "tiger"
    H[77] = "bird"
    H[31] = "cow"
    H[44] = "goat"
    H[55] = "pig"
    H[20] = "chicken"
    print("Слоты после вставки:", H.slots)
    print("Данные после вставки:", H.data)
    print("Количество элементов:", len(H))
    print()

    # 2. Получение элементов
    print("Проверка get / []:")
    print("H[20] =", H[20])
    print("H[17] =", H[17])
    print("H[99] =", H[99])  # нет такого ключа
    print()

    # 3. Проверка замены значения
    print("Замена значения по ключу 20:")
    H[20] = "duck"
    print("H[20] =", H[20])
    print()

    # 4. Проверка оператора in
    print("Проверка оператора 'in':")
    print("20 in H:", 20 in H)
    print("99 in H:", 99 in H)
    print()

    # 5. Проверка __len__
    print("Длина таблицы:", len(H))
    print()

    # 6. Добавление дополнительных элементов для вызова resize
    print("Добавляем элементы для resize:")
    H[78] = "bird11"
    H[91] = "cow11"
    H[33] = "goat11"
    H[22] = "pig11"
    H[23] = "chicken11"
    print("Размер таблицы после возможного resize:", H.size)
    print("Слоты:", H.slots)
    print("Данные:", H.data)
    print("Количество элементов:", len(H))
    print()

    # 7. Удаление элементов
    print("Удаление элементов:")
    print("До удаления 230:", H.slots)
    H[230] = "del"
    print("После добавления 230:", H.slots)
    del H[230]
    print("После удаления 230:", H.slots)
    print("Проверка get для удалённого ключа 230:", H[230])
    print("230 in H:", 230 in H)
    print("Количество элементов:", len(H))
    print()

    # 8. Удаление существующего элемента
    print("Удаление существующего ключа 20:")
    del H[20]
    print("H[20] =", H[20])
    print("20 in H:", 20 in H)
    print("Слоты после удаления 20:", H.slots)
    print("Количество элементов:", len(H))
    print()

    # 9. Проверка resize вниз (уменьшение)
    print("Удаляем несколько элементов, чтобы вызвать уменьшение таблицы:")
    del H[17]
    del H[26]
    del H[31]
    del H[44]
    del H[22]
    del H[93]
    del H[77]
    del H[55]
    del H[54]
    del H[33]
    del H[91]
    del H[78]
    print("Размер таблицы после уменьшения:", H.size)
    print("Слоты:", H.slots)
    print("Количество элементов:", len(H))
    print()



test_hash_table()


Добавляем элементы:
Слоты после вставки: [None, 93, None, 26, None, None, None, None, 77, 55, 54, None, 31, None, None, None, None, 17, None, None, 20, 44, None]
Данные после вставки: [None, 'lion', None, 'dog', None, None, None, None, 'bird', 'pig', 'cat', None, 'cow', None, None, None, None, 'tiger', None, None, 'chicken', 'goat', None]
Количество элементов: 9

Проверка get / []:
H[20] = chicken
H[17] = tiger
H[99] = None

Замена значения по ключу 20:
H[20] = duck

Проверка оператора 'in':
20 in H: True
99 in H: False

Длина таблицы: 9

Добавляем элементы для resize:
Размер таблицы после возможного resize: 23
Слоты: [22, 93, None, 26, 23, None, None, None, 77, 55, 54, 33, 31, 78, None, None, None, 17, None, None, 20, 44, 91]
Данные: ['pig11', 'lion', None, 'dog', 'chicken11', None, None, None, 'bird', 'pig', 'cat', 'goat11', 'cow', 'bird11', None, None, None, 'tiger', None, None, 'duck', 'goat', 'cow11']
Количество элементов: 14

Удаление элементов:
До удаления 230: [22, 93, None, 26

№ 2
(5 балла)
Возьмите реализацию класса HashTable из лекционных материалов и выполните
следующие доработки:
1. Переделайте существующие методы так, чтобы разрешение коллизий происходило
не при помощи концепции открытой адресации, а методом цепочек. Для этого в
каждом слоте храните связный список, реализованный классом UnorderedList из
лабораторной работы 3.
2. Реализуйте работу с функцией len (переопределите метод __len__).
3. Реализуйте работу оператора in (переопределите метод __contains__).
4. Переделайте метод put таким образом, чтобы таблица автоматически меняла размер,
когда загрузочный фактор становится больше значения 0.7. Размер должен
увеличиваться примерно в два раза до ближайшего подходящего простого числа.
5. Реализуйте работу оператора del (переопределите метод __delitem__) для удаления
элемента таблицы. Таблица должна автоматически менять размер, когда
загрузочный фактор становится меньше значения 0.2. Размер должен уменьшаться
примерно в два раза до ближайшего подходящего простого числа.
Все выполненные доработки должны быть протестированы.


In [38]:
class Node:
    def __init__(self,initdata):
        self.data = initdata
        self.next = None

    def getData(self):
        return self.data

    def getNext(self):
        return self.next

    def setData(self,newdata):
        self.data = newdata

    def setNext(self,newnext):
        self.next = newnext


class UnorderedList:
    def __init__(self):
        self.head = None

    def isEmpty(self):
        return self.head == None

    def add(self,item):
        temp = Node(item)
        temp.setNext(self.head)
        self.head = temp

    def remove(self,item):
        current = self.head
        previous = None
        found = False
        while not found:
            if current.getData() == item:
                found = True
            else:
                previous = current
                current = current.getNext()

        if previous == None:
            self.head = current.getNext()
        else:
            previous.setNext(current.getNext())

    def search(self,item):
        current = self.head
        found = False
        while current != None and not found:
            if current.getData() == item:
                found = True
            else:
                current = current.getNext()

        return found

    def size(self):
        current = self.head
        count = 0
        while current != None:
            count = count + 1
            current = current.getNext()

        return count

    def append(self,item):
        temp = Node(item)
        if self.head == None:
            self.head = temp
        else:
            current = self.head
            while current.getNext() != None:
                current = current.getNext()
            current.setNext(temp)

    def index(self, item):
        current = self.head
        pos = 0
        found = False
        while current != None and not found:
            if current.getData() == item:
                found = True
            else:
                current = current.getNext()
                pos += 1

        if found:
            return pos
        else:
            print("Позиция вне диапазона")

    def insert(self, pos, item):
        if pos < 0 or pos > self.size():
            print("Позиция вне диапазона")

        if pos == 0:
            self.add(item)
        else:
            temp = Node(item)
            current = self.head
            previous = None
            current_pos = 0

            while current_pos < pos:
                previous = current
                current = current.getNext()
                current_pos += 1

            previous.setNext(temp)
            temp.setNext(current)

    def pop(self, pos=None):
        if self.head is None:
            print("Попытка извлечения из пустого списка")

        if pos is None:
            return self.pop(self.size() - 1)
        else:
            if pos < 0 or pos >= self.size():
                print("Позиция вне диапазона")

            if pos == 0:
                item = self.head.getData()
                self.head = self.head.getNext()
                return item
            else:
                current = self.head
                previous = None
                current_pos = 0

                while current_pos < pos:
                    previous = current
                    current = current.getNext()
                    current_pos += 1

                item = current.getData()
                previous.setNext(current.getNext())
                return item

    def __str__(self):
        elements = []
        current = self.head

        while current != None:
            elements.append(str(current.getData()))
            current = current.getNext()

        return "[" + ", ".join(elements) + "]"

    def slice(self, start, stop):

        if start < 0 or stop > self.size() or start > stop:
            raise IndexError("Некорректные значения start и stop")

        result = UnorderedList()
        current = self.head
        pos = 0

        # Пропускаем элементы до start
        while pos < start:
            current = current.getNext()
            pos += 1

        # Добавляем элементы от start до stop-1
        temp_list = []
        while pos < stop:
            temp_list.append(current.getData())
            current = current.getNext()
            pos += 1

        # Добавляем элементы в обратном порядке, чтобы сохранить порядок
        for item in reversed(temp_list):
            result.add(item)

        return result


In [95]:

class Node:
    def __init__(self, initdata):
        self.data = initdata
        self.next = None

    def getData(self):
        return self.data

    def getNext(self):
        return self.next

    def setData(self, newdata):
        self.data = newdata

    def setNext(self, newnext):
        self.next = newnext


class UnorderedList:
    def __init__(self):
        self.head = None

    def isEmpty(self):
        return self.head is None

    def add(self, item):
        temp = Node(item)
        temp.setNext(self.head)
        self.head = temp

    def remove(self, item):
        current = self.head
        previous = None
        found = False
        while current and not found:
            if current.getData()[0] == item:
                found = True
            else:
                previous = current
                current = current.getNext()
        if found:
            if previous is None:
                self.head = current.getNext()
            else:
                previous.setNext(current.getNext())

    def search(self, item):
        current = self.head
        while current:
            if current.getData()[0] == item:
                return True
            current = current.getNext()
        return False

    def find(self, item):
        current = self.head
        while current:
            if current.getData()[0] == item:
                return current
            current = current.getNext()
        return None

    def __iter__(self):
        current = self.head
        while current:
            yield current.getData()
            current = current.getNext()

    def __str__(self):
        return "[" + ", ".join(str(data) for data in self) + "]"


class HashTable:
    def __init__(self, size=11):
        self.size = size
        self.slots = [UnorderedList() for _ in range(self.size)]
        self.count = 0

    def hashfunction(self, key):
        return key % self.size

    def load_factor(self):
        return self.count / self.size

    def resize(self, new_size):
        old_slots = self.slots
        self.size = new_size
        self.slots = [UnorderedList() for _ in range(self.size)]
        self.count = 0
        for slot in old_slots:
            for key, value in slot:
                self.put(key, value)

    def put(self, key, value):
        index = self.hashfunction(key)
        node = self.slots[index].find(key)
        if node:
            node.setData((key, value))  # обновление
        else:
            self.slots[index].add((key, value))
            self.count += 1

        if self.load_factor() > 0.7:
            self.resize(next_prime(self.size * 2))

    def get(self, key):
        index = self.hashfunction(key)
        node = self.slots[index].find(key)
        return node.getData()[1] if node else None

    def __getitem__(self, key):
        return self.get(key)

    def __setitem__(self, key, value):
        self.put(key, value)

    def __len__(self):
        return self.count

    def __contains__(self, key):
        index = self.hashfunction(key)
        return self.slots[index].search(key)

    def __delitem__(self, key):
        index = self.hashfunction(key)
        node = self.slots[index].find(key)
        if node:
            self.slots[index].remove(key)
            self.count -= 1
            if self.size > 11 and self.load_factor() < 0.2:
                self.resize(next_prime(max(11, self.size // 2)))
        else:
            raise KeyError(f"Key {key} not found")


H = HashTable()
H[54] = "cat"
H[93] = "lion"
H[17] = "tiger"
H[11] = "apple"
H[22] = "banana"
H[33] = "cherry"
H[20] = "chicken"


print("Все слоты:")
for i, slot in enumerate(H.slots):
    print(i, slot)

print("\nРазмер таблицы:", len(H))

print("\nДоступ к элементам:")
print(H[20])
print(H[17])

H[20] = "duck"
print(H[20])

print("\nПроверка 'in':")
print(20 in H)
print(99 in H)


H[77] = "bird"
H[44] = "goat"
print("Все слоты:")
for i, slot in enumerate(H.slots):
    print(i, slot)
print("\nРазмер таблицы:", len(H))

print("\nУдаление элемента 20")
del H[20]
print(20 in H)
del H[77]
print(77 in H)
del H[44]
print(44 in H)
del H[22]
del H[17]
del H[33]
del H[54]
print("Размер таблицы после удаления:", len(H))
for i, slot in enumerate(H.slots):
    print(i, slot)


Все слоты:
0 [(33, 'cherry'), (22, 'banana'), (11, 'apple')]
1 []
2 []
3 []
4 []
5 [(93, 'lion')]
6 [(17, 'tiger')]
7 []
8 []
9 [(20, 'chicken')]
10 [(54, 'cat')]

Размер таблицы: 7

Доступ к элементам:
chicken
tiger
duck

Проверка 'in':
True
False
Все слоты:
0 []
1 [(93, 'lion')]
2 []
3 []
4 []
5 []
6 []
7 []
8 [(54, 'cat'), (77, 'bird')]
9 []
10 [(33, 'cherry')]
11 [(11, 'apple')]
12 []
13 []
14 []
15 []
16 []
17 [(17, 'tiger')]
18 []
19 []
20 [(20, 'duck')]
21 [(44, 'goat')]
22 [(22, 'banana')]

Размер таблицы: 9

Удаление элемента 20
False
False
False
Размер таблицы после удаления: 2
0 []
1 []
2 [(93, 'lion')]
3 []
4 []
5 []
6 []
7 []
8 []
9 []
10 []
11 [(11, 'apple')]
12 []


№ 3
(2 балла)
Переделайте класс HashTable, чтобы в качестве ключей можно было использовать строки

In [96]:
class HashTable:
    def __init__(self):
        self.size = 11
        self.slots = [None] * self.size
        self.data = [None] * self.size

    def put(self, key, data):
        hashvalue = self.hashfunction(key, len(self.slots))

        if self.slots[hashvalue] is None:
            self.slots[hashvalue] = key
            self.data[hashvalue] = data
        else:
            if self.slots[hashvalue] == key:
                self.data[hashvalue] = data  # replace
            else:
                nextslot = self.rehash(hashvalue, len(self.slots))
                while self.slots[nextslot] is not None and self.slots[nextslot] != key:
                    nextslot = self.rehash(nextslot, len(self.slots))

                if self.slots[nextslot] is None:
                    self.slots[nextslot] = key
                    self.data[nextslot] = data
                else:
                    self.data[nextslot] = data  # replace

    def hashfunction(self, key, size):
        if isinstance(key, int):
            return key % size
        elif isinstance(key, str):
            #  хэш-функция для строк
            return sum(ord(c) for c in key) % size
        else:
            raise TypeError("Key must be int or str")

    def rehash(self, oldhash, size):
        return (oldhash + 1) % size

    def get(self, key):
        startslot = self.hashfunction(key, len(self.slots))

        data = None
        stop = False
        found = False
        position = startslot
        while self.slots[position] is not None and not found and not stop:
            if self.slots[position] == key:
                found = True
                data = self.data[position]
            else:
                position = self.rehash(position, len(self.slots))
                if position == startslot:
                    stop = True
        return data

    def __getitem__(self, key):
        return self.get(key)

    def __setitem__(self, key, data):
        self.put(key, data)


H = HashTable()
H["apple"] = "cat"
H["banana"] = "dog"
H["cherry"] = "lion"
H["date"] = "tiger"

print(H.slots)
print(H.data)

print(H["banana"])  # dog
H["banana"] = "duck"
print(H["banana"])  # duck
print(H["fig"])     # None


[None, None, 'apple', None, 'banana', 'cherry', None, 'date', None, None, None]
[None, None, 'cat', None, 'dog', 'lion', None, 'tiger', None, None, None]
dog
duck
None


№ 4
(2 балла)
Дана строчка русского текста, состоящая из слов и пробелов. Словом считается
последовательность русских букв, слова разделены одним или большим числом пробелов.
Для каждого слова этого текста узнайте порядковый номер его вхождения в текст именно в
той форме, в которой указано слово. Для первого вхождения слова выведите «1», для
второго вхождения того же слова выведите «2» и так далее.
Для решения этой задачи используйте класс HashTable из задания № 3.


In [97]:
text = "Раз раз раз как меня слышно Повторяю раз раз раз Повторяю"

words = text.split() 
ht = HashTable()
result = []

for word in words:
    count = ht.get(word)
    if count is None:
        count = 1
    else:
        count += 1
    ht[word] = count
    result.append(str(count))

print(" ".join(result))

1 1 2 1 1 1 1 3 4 5 2


№ 5
(3 балла)
Напишите программу, имитирующую процесс регистрации и авторизации. Для каждого
пользователя программа должна сохранять логин, хеш его пароля и «соль». Для хранения
данных можно использовать БД или файл.
Действия при сохранении пароля:
1. Сгенерируйте длинную случайную «соль» при помощи модуля secrets (secrets —
Generate secure random numbers for managing secrets — Python 3.10.0 documentation).
Длина «соли» должна быть такой же как и выходные данные используемой вами
хэш-функции. Например, если для хеширования вы используете SHA256, то на
выходе вы получите 256 бит (32 байта). В этом случае соль должна составлять не
менее 32 случайных байт.
2. Добавьте «соль» к паролю и хэшируйте его с помощью функции scrypt из модуля
hashlib (Хеширование паролей модулем hashlib в Python. (docs-python.ru)).
3. Сохраните логин, «соль» и получившейся хэш в БД или в файл.
Действия при проверке пароля:
1. Извлеките «соль» и хэш пользователя из БД или файла.
2. Добавьте «соль» к введенному паролю и хэшируйте его, используя ту же хэшфункцию, что и в алгоритме сохранения пароля.
3. Сравните получившейся хэш введенного пароля с хэшом из БД или файла. Если они
совпадают, то пароль правильный. В противном случае пароль был введен неверно.

In [8]:
import getpass
import secrets

def hash_password(password):
    salt = secrets.token_bytes(32)
    hashed = hashlib.scrypt(
        password.encode(),
        salt=salt,
        n=16384, r=8, p=1
    )
    return hashed.hex(), salt.hex()


def registration(login, password):
    hashed, salt = hash_password(password)
    with open('data.txt', 'a') as f:
        f.write(login + ' ' + hashed + ' ' + salt + '\n')


def check_password(user_password, user_login):
    with open('data.txt', 'r') as f:
        for line in f:
            login, stored_hash, stored_salt = line.strip().split()
            if user_login == login:
                new_hash = hashlib.scrypt(
                    user_password.encode(),
                    salt=bytes.fromhex(stored_salt),
                    n=16384, r=8, p=1
                ).hex()
                if new_hash == stored_hash:
                    return True
    return False

new_login = input('Введите логин: ')
new_pass = getpass.getpass('Введите пароль: ')
registration(new_login, new_pass)
user_login = input('Введите логин еще раз для проверки: ')
user_pass = getpass.getpass('Введите пароль еще раз для проверки: ')
if check_password(user_pass, user_login):
    print('Вы ввели правильный пароль')
else:
    print('Извините, но пароли не совпадают')


Вы ввели правильный пароль


№ 6
(3 балла)
Напишите программу, которая принимает от пользователя путь до директории. Для всех
файлов из данной директории должен быть вычислен хеш. Программа должна выявить и
вывести на экран все дубликаты в этой директории (т.е. файлы, у которых одинаковый хеш).

In [9]:
import hashlib
import os
from collections import Counter

file_path = 'C:/Users/lidiya/Desktop/dir'
def get_file_hash(file_path):
    hasher = hashlib.sha256()
    with open(file_path, 'rb') as file:
        buffer = file.read()
        hasher.update(buffer)
    return hasher.hexdigest()

file_list = os.listdir(file_path)
print(file_list)
hashed_files = {}
for file in file_list:
    hashed_files[file] = get_file_hash(file_path + '/' + file)

print(hashed_files)
counter = Counter(hashed_files.values())
non_unique = [item for item, count in counter.items() if count > 1]
print(non_unique)

result = {}
for value in non_unique:
    keys = [key for key, val in hashed_files.items() if val == value]
    result[value] = keys
print("Дубликаты: ", result)

['stat1.c', 'stat2.c', 'stat3 — копия.c', 'stat3.c']
{'stat1.c': '6e2a71b8896f229688e64788fdf9422478cc2298e61d0d86f09b8e003ae783ac', 'stat2.c': '4682083deb4a820f562398f63fd8a7b968b6a6381c5cc1626c597e0a04a81576', 'stat3 — копия.c': 'a7fa0d3fe1183889e58bfd914232c9b9d04705af63270208ed5e672974e4ce39', 'stat3.c': 'a7fa0d3fe1183889e58bfd914232c9b9d04705af63270208ed5e672974e4ce39'}
['a7fa0d3fe1183889e58bfd914232c9b9d04705af63270208ed5e672974e4ce39']
Дубликаты:  {'a7fa0d3fe1183889e58bfd914232c9b9d04705af63270208ed5e672974e4ce39': ['stat3 — копия.c', 'stat3.c']}
